In [6]:
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import numpy as np

In [7]:
import sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

db_path = next(
    (
        parent / "2_IPL_Cricket" / "ipl.db"
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "2_IPL_Cricket" / "ipl.db").is_file()
    ),
    None,
)
if db_path is None:
    raise FileNotFoundError(
        f"Could not find 2_IPL_Cricket/ipl.db from the current directory: {Path.cwd()}"
    )

con = sqlite3.connect(db_path)

def q(sql):
    return pd.read_sql_query(sql, con)

In [8]:
q(""" SELECT COUNT(*) AS n,
        SUM(CASE WHEN toss_winner = match_winner
                 THEN 1 ELSE 0 END)AS k
FROM matches_clean
WHERE result='win';""")

,n,k
0,1187,613


syntax= stats.binomtest(k,n,p)
k=no of success
n= total trail,data
p=expected probability



In [9]:
# if pvalue<sv then significant other wise not significant
p=stats.binomtest(613,1187,0.5).pvalue
ci=proportion_confint(613,1187, method='wilson')
print(p)
print(ci)

0.2700389712415702
(0.48799203833275506, 0.5447579131499739)


Since the p-value (0.2700) is greater than the significance level (0.05), we fail to reject the null hypothesis. There is insufficient statistical evidence to conclude that the true population proportion is different from 50% in the analyzed data.

In [10]:
#Does the decision taken after the toss matter?
#h0- Captains who choose to bat and captains who 
# choose to field win equally often. The gap we saw is luck.
#h1-The two choices do not win equally often.

q(""" SELECT toss_decision,
        SUM(CASE WHEN toss_winner = match_winner
                 THEN 1 ELSE 0 END ) AS won,
        SUM(CASE WHEN toss_winner <> match_winner
                 THEN 1 ELSE 0 END) AS lost
FROM matches_clean
WHERE result = 'win'
GROUP BY toss_decision;""")

,toss_decision,won,lost
0,bat,184,218
1,field,429,356


In [11]:
# if pvalue<sv then significant other wise not significant
table =[[184,218],[429,356]]
chi2, p, dof, expected = stats.chi2_contingency(table)
print(round(chi2,2),round(p,4))

8.04 0.0046


Since the p-value  is less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence of an association between the toss decision and the match outcome in the analyzed data.

In [12]:
#Does the chasing team really have an advantage?
#h0-Both sides win half the time. The 54.51% we saw is luck.
#h1-A chase does not succeed half the time.

q(""" SELECT COUNT(*) AS n,
        SUM(chase_won) AS k
FROM v_match_totals;""")

,n,k
0,1212,633


In [13]:
r=stats.binomtest(647,1187,0.5).pvalue
ri=proportion_confint(647,1187, method='wilson')
print(r)
print(ri)

0.002080449027018435
(0.5166432360570298, 0.5732091949340473)


Since the p-value is less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence that the chasing success rate is different from 50% in the analyzed data. The 95% Wilson confidence interval (51.66%–57.31%) further supports this finding.

In [14]:
#Does the chase win rate really fall as the target rises?

# h0-The chase win rate is the same whatever the target. 
# The differences between the bands are luck.

#h1- The chase win rate is not the same in every target band.

q(""" SELECT CASE WHEN first_innings_runs<140 THEN 'under 140'
             WHEN first_innings_runs <160 THEN '140-159'
             WHEN first_innings_runs <180 THEN '160-179'
             WHEN first_innings_runs <200 THEN '180-199'
             ELSE '200+' END AS band,
        SUM(chase_won) AS won,
        SUM(1-chase_won) AS lost
FROM  v_match_totals
GROUP BY band;""")

,band,won,lost
0,140-159,167,81
1,160-179,166,149
2,180-199,82,139
3,200+,45,156
4,under 140,173,54


In [15]:
table =[[171,70],[166,146],[83,134],[46,152],[181,38]]
chi2, p, dof, expected = stats.chi2_contingency(table)
print(round(chi2,2))
print(p)

197.68
1.1854224475712924e-41


Since the p-value (1.1854 × 10⁻⁴¹) is far less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence that the chase win rate differs across the target bands in the analyzed data. The results show that the chase win rate decreases substantially as the target increases.

In [16]:
#  are defended totals actually bigger than chased ones?
# h0 both kinds of total average the same . the gap we see is luck
# h1 the two averages are not equal.

